# Redrob Hackathon — Kaggle Dual-GPU Pipeline

**Goal:** Rank top 100 candidates from 100K for Senior AI Engineer.

**Kaggle setup:** Enable **GPU T4 x2** under *Settings → Accelerator*.

### Steps
1. Clone repo + extract dataset
2. Install deps
3. Pre-compute embeddings (dual GPU) + features
4. Rank (CPU, <3 min)
5. Validate + download

## Step 1 — Clone Repo + Extract Dataset

In [ ]:
# Clone the repository
!git clone https://github.com/JASHWANTHS07/india-runs.git
%cd india-runs
!ls

In [ ]:
# Install unrar and extract Data.rar
!apt-get install -qq unrar
!unrar x -o+ Data/Data.rar Data/
print("Extraction complete.")
!ls Data/India_runs_data_and_ai_challenge/

In [ ]:
# Verify dataset
import os
CANDIDATES_PATH = "Data/India_runs_data_and_ai_challenge/candidates.jsonl"
size_mb = os.path.getsize(CANDIDATES_PATH) / 1e6
print(f"candidates.jsonl: {size_mb:.1f} MB")
assert size_mb > 400, f"File too small ({size_mb:.1f} MB)"
print("Dataset OK")

## Step 2 — Install Dependencies + GPU Check

In [ ]:
!pip install -q sentence-transformers tqdm pandas pyarrow

import torch
gpu_count = torch.cuda.device_count()
for i in range(gpu_count):
    name = torch.cuda.get_device_name(i)
    mem = torch.cuda.get_device_properties(i).total_mem / 1e9
    print(f"  GPU {i}: {name} ({mem:.1f} GB)")
print(f"\nTotal GPUs: {gpu_count}")

## Step 3 — Pre-compute Embeddings (Dual GPU) + Features

Embeds 100K candidates using `BAAI/bge-large-en-v1.5` with DataParallel across both T4 GPUs, then extracts 45+ structured features.

**Expected time:** ~15–25 min on 2× T4.

In [ ]:
import json, sys, time, os
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

CANDIDATES_PATH = "Data/India_runs_data_and_ai_challenge/candidates.jsonl"
ARTIFACTS = "./artifacts"
os.makedirs(ARTIFACTS, exist_ok=True)
BATCH_SIZE = 256

# ---- Load candidates ----
print("Loading candidates...")
candidates = []
with open(CANDIDATES_PATH, 'r') as f:
    for line in f:
        candidates.append(json.loads(line))
print(f"Loaded {len(candidates)} candidates")

# ---- Extract features ----
print("\nExtracting features...")
sys.path.insert(0, '.')
from src.features import extract_features
from dataclasses import asdict

feat_dicts = []
for cand in tqdm(candidates, desc="Extracting features"):
    feat_dicts.append(asdict(extract_features(cand)))

feat_df = pd.DataFrame(feat_dicts)
feat_df.to_parquet(os.path.join(ARTIFACTS, "features.parquet"), index=False)
print(f"Features saved: {len(feat_df)} rows, {len(feat_df.columns)} columns")

# ---- Build texts for embedding ----
all_texts = [d.get("profile_text", "") for d in feat_dicts]
print(f"\nTexts to embed: {len(all_texts)}")

# ---- Load model + DataParallel ----
print("Loading embedding model...")
model = SentenceTransformer("BAAI/bge-large-en-v1.5")
gpu_count = torch.cuda.device_count()
print(f"Wrapping model across {gpu_count} GPU(s)...")
if gpu_count > 1:
    model._first_module().auto_model = torch.nn.DataParallel(
        model._first_module().auto_model
    )

# ---- Embed JD ----
from src.precompute import JD_TEXT
jd_emb = model.encode([JD_TEXT], normalize_embeddings=True, device='cuda:0', batch_size=1)
np.save(os.path.join(ARTIFACTS, "jd_embedding.npy"), jd_emb[0])
print("JD embedding saved.")

# ---- Embed candidates with progress bar ----
print(f"\nEmbedding {len(all_texts)} candidates across {gpu_count} GPU(s)...")
num_batches = (len(all_texts) + BATCH_SIZE - 1) // BATCH_SIZE
all_embeddings = []
t0 = time.time()

for i in tqdm(range(0, len(all_texts), BATCH_SIZE), total=num_batches, desc="Embedding batches"):
    batch = all_texts[i : i + BATCH_SIZE]
    emb = model.encode(batch, normalize_embeddings=True, show_progress_bar=False,
                       device='cuda:0', batch_size=BATCH_SIZE)
    all_embeddings.append(emb)

embeddings = np.vstack(all_embeddings).astype(np.float32)
elapsed = time.time() - t0
np.save(os.path.join(ARTIFACTS, "embeddings.npy"), embeddings)
print(f"\nEmbeddings saved: {embeddings.shape} in {elapsed:.1f}s")

In [ ]:
# Verify artifacts
emb = np.load('artifacts/embeddings.npy')
jd  = np.load('artifacts/jd_embedding.npy')
df  = pd.read_parquet('artifacts/features.parquet')

print(f"embeddings.npy  : {emb.shape}  ({emb.nbytes/1e6:.0f} MB)")
print(f"jd_embedding    : {jd.shape}")
print(f"features.parquet: {len(df)} rows, {len(df.columns)} cols")
assert emb.shape == (100000, 1024)
assert jd.shape == (1024,)
assert len(df) == 100000
print("Artifacts OK")

## Step 4 — Rank Candidates (CPU)

Scores all 100K candidates and selects top 100. **<3 min, no GPU needed.**

In [ ]:
!python src/rank.py \
    --artifacts ./artifacts \
    --out       ./jashwanth_s.csv

## Step 5 — Validate & Download

In [ ]:
import pandas as pd, sys
from dataclasses import fields

df = pd.read_csv('jashwanth_s.csv')
assert len(df) == 100
assert set(df['rank']) == set(range(1, 101))
scores = df.sort_values('rank')['score'].tolist()
assert all(scores[i] >= scores[i+1] for i in range(len(scores)-1))

sys.path.insert(0, '.')
from src.features import CandidateFeatures
from src.honeypot import is_honeypot

feat_df = pd.read_parquet('artifacts/features.parquet')
top_feats = feat_df[feat_df['candidate_id'].isin(set(df['candidate_id']))]
honeypots = []
for _, row in top_feats.iterrows():
    kwargs = {f.name: row[f.name] for f in fields(CandidateFeatures) if f.name in row.index}
    kwargs.setdefault('profile_text', '')
    if is_honeypot(CandidateFeatures(**kwargs)):
        honeypots.append(row['candidate_id'])

print(f"Rows: {len(df)} | Scores: {scores[0]:.4f} → {scores[-1]:.4f} | Honeypots: {len(honeypots)}")
assert len(honeypots) == 0
print("All checks passed.")

In [ ]:
# Inspect top 10
pd.set_option('display.max_colwidth', 80)
df.sort_values('rank').head(10)[['rank', 'candidate_id', 'score', 'reasoning']]

In [ ]:
# Download
from IPython.display import FileLink
FileLink('jashwanth_s.csv')